# LoRA Applied: How It Fits

In [1]:
!pip install -q transformers peft datasets accelerate

In [6]:
from datasets import load_dataset

# Download the IMDb dataset
dataset = load_dataset("stanfordnlp/imdb")

# Verify we have train and test splits
print(f"Training samples: {len(dataset['train'])}")
print(f"Test samples: {len(dataset['test'])}")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Training samples: 25000
Test samples: 25000


In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    result = tokenizer(batch["text"], padding="max_length", truncation=True, max_length=512)
    result["labels"] = batch["label"]  # Keep the label for training
    return result

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [8]:
dataset = dataset.map(tokenize, batched=True)

# Set format for PyTorch

dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [39]:
from transformers import AutoModelForSequenceClassification


model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)

# Count trainable parameters in baseline

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable} / {total} ({100*trainable/total:.2f}%)")
# Expected: Trainable: ~67M / 67M (100%)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable: 66955010 / 66955010 (100.00%)


# Efficient Fine-Tuning with LoRA: Training and Evaluation in Practice

In [14]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,                              # Rank: smaller = fewer params
    lora_alpha=16,                    # Scaling factor for LoRA updates
    target_modules=["q_lin", "v_lin"], # DistilBERT attention layers
    lora_dropout=0.1,                 # Regularization
    bias="none",                      # Don't add bias parameters
    task_type="SEQ_CLS"               # Sequence classification task
)

In [15]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 7.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [16]:
# Attach LoRA adapters to the model

lora_model = get_peft_model(model, lora_config)

# Show the dramatic reduction in trainable parameters

lora_model.print_trainable_parameters()


trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


In [17]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",      # Check performance after each epoch
    num_train_epochs=1,               # Just 1 epoch for demo
    per_device_train_batch_size=8,   # 8 examples per batch
    per_device_eval_batch_size=8,
    logging_steps=50,                 # Log every 50 steps
    save_steps=2000,                  # Save checkpoint every 2000 steps
    learning_rate=5e-4,               # Higher than typical (faster demo)
    weight_decay=0.01,                # L2 regularization
)

# Create small train/eval subsets

train_subset = dataset["train"].shuffle(seed=42).select(range(2000))
eval_subset = dataset["test"].shuffle(seed=42).select(range(500))

# Initialize the Trainer

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_subset,
    eval_dataset=eval_subset,
)

# Run the training

print("Starting training with LoRA adapters...")
trainer.train()


Starting training with LoRA adapters...


Epoch,Training Loss,Validation Loss
1,0.277554,0.295019


TrainOutput(global_step=250, training_loss=0.37956034851074216, metrics={'train_runtime': 86.338, 'train_samples_per_second': 23.165, 'train_steps_per_second': 2.896, 'total_flos': 269478813696000.0, 'train_loss': 0.37956034851074216, 'epoch': 1.0})

In [38]:
import torch

text = "This movie was an absolute masterpiece. Brilliant performances!"

# Tokenize the input text

inputs = tokenizer(text, return_tensors="pt")

# Move inputs to same device as models

device = next(model.parameters()).device
inputs = {k: v.to(device) for k, v in inputs.items()}

# Get predictions from both models

with torch.no_grad():  # Don't compute gradients during inference
    baseline_logits = model(**inputs).logits
    lora_logits = lora_model(**inputs).logits

# Convert logits to probabilities

baseline_probs = baseline_logits.softmax(-1)[0].cpu().numpy()
lora_probs = lora_logits.softmax(-1)[0].cpu().numpy()

print(f"Input: {text}")
print(f"\nBaseline predictions:")
print(f"  Negative: {baseline_probs[0]:.1%}")
print(f"  Positive: {baseline_probs[1]:.1%}")
print(f"\nLoRA fine-tuned predictions:")
print(f"  Negative: {lora_probs[0]:.1%}")
print(f"  Positive: {lora_probs[1]:.1%}")

Input: This movie was an absolute masterpiece. Brilliant performances!

Baseline predictions:
  Negative: 0.2%
  Positive: 99.8%

LoRA fine-tuned predictions:
  Negative: 0.2%
  Positive: 99.8%
